# ☕ 咖啡豆瑕疵檢測模型訓練

使用 YOLOv8 分類模式（Image Classification）

**前置準備：**
1. 將 `data/` 資料夾上傳到 Google Drive 或這裡
2. 確認資料結構：
```
data/
  train/
    good/
    broken/
    black/
    moldy/
    stink/
  val/
    good/
    broken/
    black/
    moldy/
    stink/
```

In [ ]:
# 安裝 ultralytics
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import os

# 設定資料路徑（請修改為你的實際路徑）
DATA_DIR = '/content/coffee-bean-inspector/data'

# 確認類別
classes = ['good', 'broken', 'black', 'moldy', 'stink']
for c in classes:
    train_path = f'{DATA_DIR}/train/{c}'
    val_path = f'{DATA_DIR}/val/{c}'
    if os.path.exists(train_path):
        print(f'{c}: train={len(os.listdir(train_path))}, val={len(os.listdir(val_path)) if os.path.exists(val_path) else 0}')
    else:
        print(f'⚠️  類別 {c} 資料夾不存在：{train_path}')

In [ ]:
# 訓練分類模型（YOLOv8n-cls，輕量）
# 若 GPU 記憶體夠，可換成 yolov8s-cls 或 yolov8m-cls
model = YOLO('yolov8n-cls.pt')  # 預訓練模型

results = model.train(
    data=DATA_DIR,
    epochs=50,            # 可根據資料量調整
    model='yolov8n-cls',  # 輕量模型，樹莓派推論快
    imgsz=224,            # 統一尺寸，224 足夠
    patience=10,          # 早停
    batch=32,
    project='/content/runs',
    name='coffee_classify',
    exist_ok=True,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    verbose=True
)
print('訓練完成！')

In [ ]:
# 評估模型
from ultralytics import YOLO

best_model = YOLO('/content/runs/coffee_classify/weights/best.pt')

# 驗證集表現
val_results = best_model.val(data=DATA_DIR, verbose=True)
print(f"Top-1 準確率：{val_results.top1 * 100:.1f}%")
print(f"Top-5 準確率：{val_results.top5 * 100:.1f}%")

In [ ]:
# 匯出模型為 ONNX / TFLite（樹莓派可用的格式）
# ONNX（推薦，支援廣）
best_model.export(format='onnx', imgsz=224)
print("已匯出 ONNX 模型")

# 或 TFLite（行動裝置/邊緣裝置）
best_model.export(format='tflite', imgsz=224)
print("已匯出 TFLite 模型")

In [ ]:
# 下載模型到本地
from google.colab import files
files.download('/content/runs/coffee_classify/weights/best.pt')
print("模型已下載，請放至樹莓派 pi/models/best.pt")